# Kerry-3D — Seismic Data Starter Notebook

**Dataset:** Kerry-3D (New Zealand, Taranaki Basin)  
**Source:** http://s3.amazonaws.com/open.source.geoscience/open_data/newzealand/Taranaiki_Basin/Keri_3D/Kerry3D.segy  
**Format:** SEG-Y  
**License:** Public domain (open geoscience data)

### Volume specs (discovered from header scan)
| Property | Value |
|---|---|
| Inline byte | **223** (non-standard — not the default 189) |
| Crossline byte | **21** (CDP field — not the default 193) |
| Inlines | 510 → 796  (287 lines) |
| Crosslines | 58 → 792  (735 lines) |
| Time samples | 1252  (0 → 5004 ms, 4 ms interval) |
| Volume shape | (287, 735, 1252) |
| File size | ~849 MB |

---
**Requirements:**
```
pip install segyio numpy matplotlib
```

## 0. Configuration

In [ ]:
import os

# Path to Kerry3D SEG-Y file — update to your AWS mount path
SEGY_PATH = "../data/Teapot_dome.sgy"

# Kerry-3D uses non-standard header byte locations
ILINE_BYTE = 223   # inline number is stored at byte 223
XLINE_BYTE = 21    # crossline number is stored at byte 21 (CDP field)

FIG_DIR = "./figures"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"SEG-Y path : {SEGY_PATH}")
print(f"File exists: {os.path.exists(SEGY_PATH)}")
print(f"File size  : {os.path.getsize(SEGY_PATH)/1e6:.1f} MB")

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import segyio

print(f"segyio {segyio.__version__}  |  numpy {np.__version__}")

## 2. Inspect metadata

In [ ]:
with segyio.open(SEGY_PATH, iline=ILINE_BYTE, xline=XLINE_BYTE) as f:
    inlines    = f.ilines.copy()
    crosslines = f.xlines.copy()
    t_axis     = f.samples.copy()        # time in ms
    n_samples  = f.samples.size
    dt_ms      = segyio.dt(f) / 1000.0  # microseconds -> ms

print("=== Kerry-3D Volume Metadata ===")
print(f"  Inlines    : {inlines[0]} -> {inlines[-1]}   ({len(inlines)} lines)")
print(f"  Crosslines : {crosslines[0]} -> {crosslines[-1]}   ({len(crosslines)} lines)")
print(f"  Time       : {t_axis[0]:.0f} ms -> {t_axis[-1]:.0f} ms   ({n_samples} samples @ {dt_ms:.0f} ms)")
print(f"  Volume shape: ({len(inlines)}, {len(crosslines)}, {n_samples})")
print(f"  Est. RAM   : {len(inlines)*len(crosslines)*n_samples*4/1e6:.0f} MB")

## 3. Load full volume

In [ ]:
with segyio.open(SEGY_PATH, iline=ILINE_BYTE, xline=XLINE_BYTE) as f:
    volume = segyio.tools.cube(f)   # shape: (n_inlines, n_xlines, n_samples)

print(f"Loaded: shape={volume.shape}  dtype={volume.dtype}")
print(f"Amplitude: min={volume.min():.3f}  max={volume.max():.3f}  std={volume.std():.4f}")

# Clip percentiles for display
vmin = np.percentile(volume, 2)
vmax = np.percentile(volume, 98)
print(f"Display clip: [{vmin:.3f}, {vmax:.3f}]  (2nd-98th percentile)")

## 4. Three-slice overview — Inline / Crossline / Time

In [ ]:
mid_il = volume.shape[0] // 2
mid_xl = volume.shape[1] // 2
mid_t  = volume.shape[2] // 2

fig = plt.figure(figsize=(18, 5))
fig.suptitle(
    f"Kerry-3D  |  shape={volume.shape}  |  "
    f"inline {inlines[mid_il]} / xline {crosslines[mid_xl]} / {t_axis[mid_t]:.0f} ms",
    fontsize=12, fontweight="bold"
)
gs = gridspec.GridSpec(1, 3, wspace=0.3)

# Inline slice
ax0 = fig.add_subplot(gs[0])
ax0.imshow(
    volume[mid_il, :, :].T,
    cmap="gray", aspect="auto", vmin=vmin, vmax=vmax,
    extent=[crosslines[0], crosslines[-1], t_axis[-1], t_axis[0]]
)
ax0.set_title(f"Inline {inlines[mid_il]}", fontsize=11)
ax0.set_xlabel("Crossline"); ax0.set_ylabel("Time (ms)")

# Crossline slice
ax1 = fig.add_subplot(gs[1])
ax1.imshow(
    volume[:, mid_xl, :].T,
    cmap="gray", aspect="auto", vmin=vmin, vmax=vmax,
    extent=[inlines[0], inlines[-1], t_axis[-1], t_axis[0]]
)
ax1.set_title(f"Crossline {crosslines[mid_xl]}", fontsize=11)
ax1.set_xlabel("Inline"); ax1.set_ylabel("Time (ms)")

# Time slice
ax2 = fig.add_subplot(gs[2])
ax2.imshow(
    volume[:, :, mid_t],
    cmap="gray", aspect="auto", vmin=vmin, vmax=vmax,
    extent=[crosslines[0], crosslines[-1], inlines[-1], inlines[0]]
)
ax2.set_title(f"Time slice {t_axis[mid_t]:.0f} ms", fontsize=11)
ax2.set_xlabel("Crossline"); ax2.set_ylabel("Inline")

plt.savefig(f"{FIG_DIR}/kerry3d_overview.png", dpi=150, bbox_inches="tight")
print("Saved: kerry3d_overview.png")
plt.show()

## 5. Chair Plot — 3D corner visualization

A **chair plot** cuts the seismic cube at three orthogonal planes and displays them in a 3D perspective view — like looking at the corner of an open box.  
It gives an immediate feel for the 3D structure of reflectors and any faults.

We generate 3 plots at different cut positions (shallow / mid-depth / deep).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401

norm_chair = plt.Normalize(vmin=vmin, vmax=vmax)
cmap_chair = plt.cm.gray

def chair_plot(volume, il_idx, xl_idx, t_idx, inlines, crosslines, t_axis,
               title="", stride=3):
    """
    3D chair/corner plot of a seismic volume.
    Cuts at inline=il_idx, crossline=xl_idx, time=t_idx.
    stride: downsampling factor for display speed (3-4 is good).
    """
    fig = plt.figure(figsize=(11, 8))
    ax  = fig.add_subplot(111, projection='3d')

    xl_ds = crosslines[::stride]
    il_ds = inlines[::stride]
    t_ds  = t_axis[::stride]

    # ── Inline face: fixed inline, crossline x time ──
    XL, T = np.meshgrid(xl_ds, t_ds, indexing='ij')
    ax.plot_surface(
        np.full_like(XL, inlines[il_idx]), XL, T,
        facecolors=cmap_chair(norm_chair(volume[il_idx, ::stride, ::stride])),
        shade=False, rstride=1, cstride=1, antialiased=False
    )

    # ── Crossline face: fixed crossline, inlines up to cut x time ──
    il_cut  = il_ds[il_ds <= inlines[il_idx]]
    il_idxs = np.searchsorted(inlines, il_cut)
    IL2, T2 = np.meshgrid(il_cut, t_ds, indexing='ij')
    ax.plot_surface(
        IL2, np.full_like(IL2, crosslines[xl_idx]), T2,
        facecolors=cmap_chair(norm_chair(volume[il_idxs, xl_idx, ::stride])),
        shade=False, rstride=1, cstride=1, antialiased=False
    )

    # ── Time face: fixed time, inlines up to cut x crosslines from cut onward ──
    xl_cut  = xl_ds[xl_ds >= crosslines[xl_idx]]
    xl_idxs = np.searchsorted(crosslines, xl_cut)
    IL3, XL3 = np.meshgrid(il_cut, xl_cut, indexing='ij')
    ax.plot_surface(
        IL3, XL3, np.full_like(IL3, t_axis[t_idx]),
        facecolors=cmap_chair(norm_chair(volume[np.ix_(il_idxs, xl_idxs, [t_idx])][:, :, 0])),
        shade=False, rstride=1, cstride=1, antialiased=False
    )

    ax.set_xlabel('Inline',    labelpad=8)
    ax.set_ylabel('Crossline', labelpad=8)
    ax.set_zlabel('Time (ms)', labelpad=8)
    ax.invert_zaxis()          # time increases downward
    ax.view_init(elev=25, azim=-60)
    ax.set_title(
        f"Kerry-3D  Chair Plot\n"
        f"Cut: inline={inlines[il_idx]}  xline={crosslines[xl_idx]}  t={t_axis[t_idx]:.0f} ms"
        + (f"  |  {title}" if title else ""),
        fontsize=11, fontweight='bold', pad=12
    )
    plt.tight_layout()
    return fig


# 3 chair plots at different depth cuts
cuts = [
    (len(inlines)//2,    len(crosslines)//2,    len(t_axis)//3,    "shallow"),
    (len(inlines)//3,    len(crosslines)*2//3,  len(t_axis)//2,    "mid-depth"),
    (len(inlines)*2//3,  len(crosslines)//3,    len(t_axis)*2//3,  "deep"),
]

for il_i, xl_i, t_i, label in cuts:
    fig = chair_plot(volume, il_i, xl_i, t_i, inlines, crosslines, t_axis,
                     title=label, stride=3)
    out = f"{FIG_DIR}/kerry3d_chair_{label}.png"
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f"Saved: {out}")
    plt.show()

## 5. Amplitude histogram

In [ ]:
flat   = volume.ravel()
sample = flat[np.random.choice(len(flat), size=len(flat)//100, replace=False)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample, bins=200, color="steelblue", alpha=0.8, edgecolor="none")
ax.axvline(vmin, color="red",    linestyle="--", label=f"2nd pct  ({vmin:.1f})")
ax.axvline(vmax, color="orange", linestyle="--", label=f"98th pct ({vmax:.1f})")
ax.set_title("Kerry-3D Amplitude Distribution", fontsize=12)
ax.set_xlabel("Amplitude"); ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/kerry3d_histogram.png", dpi=150, bbox_inches="tight")
print("Saved: kerry3d_histogram.png")
plt.show()

## 6. Load a single inline (memory-efficient for large datasets)

In [ ]:
target_inline = inlines[mid_il]

with segyio.open(SEGY_PATH, iline=ILINE_BYTE, xline=XLINE_BYTE) as f:
    il_slice = f.iline[target_inline]   # shape: (n_xlines, n_samples)

print(f"Loaded inline {target_inline}  -> shape {il_slice.shape}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(
    il_slice.T, cmap="gray", aspect="auto", vmin=vmin, vmax=vmax,
    extent=[crosslines[0], crosslines[-1], t_axis[-1], t_axis[0]]
)
ax.set_title(f"Kerry-3D — Inline {target_inline}  (single inline load)", fontsize=12)
ax.set_xlabel("Crossline"); ax.set_ylabel("Time (ms)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/kerry3d_inline_{target_inline}.png", dpi=150, bbox_inches="tight")
print(f"Saved: kerry3d_inline_{target_inline}.png")
plt.show()

## 7. Extract a sub-cube patch (for model input)

In [ ]:
# Crop a 128 x 128 x 128 patch from the middle of the volume
il_c = mid_il
xl_c = mid_xl
t_c  = volume.shape[2] // 3   # shallower time window avoids empty zone

patch = volume[
    il_c - 64 : il_c + 64,
    xl_c - 64 : xl_c + 64,
    t_c  - 64 : t_c  + 64
]
print(f"Patch shape: {patch.shape}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f"Kerry-3D sub-cube {patch.shape} — mid slices", fontsize=12, fontweight="bold")
kw = dict(cmap="gray", aspect="auto",
          vmin=np.percentile(patch, 2), vmax=np.percentile(patch, 98))

axes[0].imshow(patch[patch.shape[0]//2, :, :].T, **kw)
axes[0].set_title("Inline slice"); axes[0].set_xlabel("Crossline"); axes[0].set_ylabel("Time")

axes[1].imshow(patch[:, patch.shape[1]//2, :].T, **kw)
axes[1].set_title("Crossline slice"); axes[1].set_xlabel("Inline"); axes[1].set_ylabel("Time")

axes[2].imshow(patch[:, :, patch.shape[2]//2], **kw)
axes[2].set_title("Time slice"); axes[2].set_xlabel("Crossline"); axes[2].set_ylabel("Inline")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/kerry3d_subcube.png", dpi=150, bbox_inches="tight")
print("Saved: kerry3d_subcube.png")
plt.show()

## 8. Save volume as NumPy for fast re-loading

In [ ]:
npy_path = f"{FIG_DIR}/kerry3d_volume.npy"
np.save(npy_path, volume.astype(np.float32))
print(f"Saved {npy_path}  ({os.path.getsize(npy_path)/1e6:.0f} MB)")

# Verify round-trip
v2 = np.load(npy_path)
print(f"Reload: shape={v2.shape}  max_diff={np.max(np.abs(v2 - volume)):.6f}")

---
## Summary

| Step | What you did |
|---|---|
| Header scan | Found iline=byte 223, xline=byte 21 (non-standard for Kerry-3D) |
| `segyio.open` | Opened with correct byte locations |
| `segyio.tools.cube` | Loaded full 3D volume (287 × 735 × 1252) |
| 3-slice plot | Inline / crossline / time slice visualization |
| Histogram | Amplitude distribution + display clip range |
| Single inline | Memory-efficient single-inline load |
| Sub-cube | 128³ patch crop for model input |
| `.npy` export | Fast-reload format |

**Key takeaway:** Always scan trace headers first — real-world SEG-Y files often use non-standard byte locations for inline/crossline numbers.